In [ ]:
import sys
import torch
import numpy as np
sys.path.append('../')
from GeCoSleep import *
from baselines import *

In [ ]:
from tqdm import tqdm
from sklearn.cluster import kmeans_plusplus

num_tasks = 4
num_categories = 5
num_samples_per_class = 200

datas: dict[str, np.ndarray] = {}
for t in range(num_tasks):
    feature = torch.load(f'./datacache/task{t}_feature.pt', map_location='cpu', weights_only=True)
    label = torch.load(f'./datacache/task{t}_label.pt', map_location='cpu', weights_only=True)
    gen_feature = torch.load(f'./datacache/task{t}_gen_feature.pt', map_location='cpu', weights_only=True)
    untrained_gen_feature = torch.load(f'./datacache/task{t}_untrained_gen_feature.pt', map_location='cpu', weights_only=True)
    num_samples, seq_length, embeddings = feature.shape
    feature = feature.view(num_samples * seq_length, -1)
    label = label.view(num_samples * seq_length)
    gen_feature = gen_feature.view(num_samples * seq_length, -1)
    untrained_gen_feature = untrained_gen_feature.view(num_samples * seq_length, -1)
    for c in range(num_categories):
        datas[f'task{t}_class{c}_feature'] = feature[label == c].cpu().numpy()
        datas[f'task{t}_class{c}_gen_feature'] = gen_feature[label == c].cpu().numpy()
        datas[f'task{t}_class{c}_untrained_gen_feature'] = untrained_gen_feature[label == c].cpu().numpy()
for (key, value) in datas.items():
    print(key, value.shape)

In [ ]:
data_sampled: dict[str, np.ndarray] = {}
for (key, value) in tqdm(datas.items()):
    centers, _ = kmeans_plusplus(value, num_samples_per_class)
    data_sampled[key] = centers
for (key, value) in data_sampled.items():
    print(key, value.shape)

In [ ]:
np.savez('./datacache/sampled_features.npz', **data_sampled)